In [ ]:
# %%
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import json
import anndata as ad
import scipy.sparse as sp
from datetime import datetime
import joblib
import warnings

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score

warnings.filterwarnings("ignore", category=UserWarning)



In [ ]:
# ============================================================
# IO + Alignment utilities (same style as your training code)
# ============================================================

def _assert_aligned_obs(*adatas: ad.AnnData):
    base = adatas[0].obs_names.astype(str).to_numpy()
    for k, A in enumerate(adatas[1:], start=1):
        cur = A.obs_names.astype(str).to_numpy()
        if not np.array_equal(base, cur):
            raise ValueError(
                f"obs_names mismatch between modality 0 and modality {k}. "
                f"Example: {base[:3]} vs {cur[:3]}"
            )

def get_norm(adata: ad.AnnData):
    X = adata.layers["norm"] if "norm" in adata.layers else adata.X
    return X.tocsr() if sp.issparse(X) else np.asarray(X)

def load_regression_data(target_name: str, base_dir: str = "../data") -> dict:
    """
    Loads RNA, ATAC, ADT-minus; loads y from response CSV and aligns by cell IDs.
    (This matches your training loader.)
    """
    data_path = Path(base_dir)
    resp_path = data_path / "response"

    rna = ad.read_h5ad(data_path / "rna.h5ad")
    atac = ad.read_h5ad(data_path / "atac.h5ad")
    adt_minus = ad.read_h5ad(data_path / f"adt_minus_{target_name}.h5ad")

    _assert_aligned_obs(rna, atac, adt_minus)

    cell_ids = rna.obs_names.astype(str)
    y_df = pd.read_csv(resp_path / f"{target_name}.csv", index_col=0)

    # Align y by cell IDs (CRITICAL)
    y_vec = y_df.loc[cell_ids].iloc[:, 0].to_numpy(dtype=float)

    return {"rna": rna, "atac": atac, "adt_minus": adt_minus, "y": y_vec, "cell_ids": cell_ids}

def load_indices_csv_robust(
    path: Path,
    n_cells: int,
    *,
    default_base: str = "0-based",   # "0-based" or "1-based"
    dedup: bool = False,             # keep order by default
) -> np.ndarray:
    """
    Load a single numeric index column robustly.

    Key design (matches your R helper):
      - If we see exact sentinel range 0..n_cells-1 => 0-based
      - Else if we see exact sentinel range 1..n_cells => convert to 0-based
      - Else => fall back to default_base (you want 0-based)

    Safety:
      - NO clipping (fail fast if out of range)
      - NO sorting/unique unless dedup=True (order matters for alignment)
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")

    # Read with header, fallback to no-header
    df = pd.read_csv(path)
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        df = pd.read_csv(path, header=None)
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            raise ValueError(f"No numeric column found in {path}")

    idx = df[num_cols[0]].to_numpy()

    if idx.size == 0:
        raise ValueError(f"Empty index file: {path}")

    # Require integer-valued (no silent truncation)
    if np.isnan(idx).any():
        bad = np.where(np.isnan(idx))[0][:10]
        raise ValueError(f"NaNs in {path} at rows {bad.tolist()}")

    idx_int = idx.astype(np.int64)
    if not np.allclose(idx, idx_int):
        bad = np.where(~np.isclose(idx, idx_int))[0][:10]
        raise ValueError(f"Non-integer indices in {path} at rows {bad.tolist()}: {idx[bad].tolist()}")

    idx = idx_int
    mn, mx = int(idx.min()), int(idx.max())

    # Decide base using sentinel matches first (R-style)
    if mn == 0 and mx == (n_cells - 1):
        idx0 = idx
        mode = "0-based (sentinel match)"
    elif mn == 1 and mx == n_cells:
        idx0 = idx - 1
        mode = "1-based (sentinel match) -> converted"
    else:
        # Ambiguous / partial range: follow default_base (you want 0-based)
        if default_base == "0-based":
            idx0 = idx
            mode = "0-based (default)"
        elif default_base == "1-based":
            idx0 = idx - 1
            mode = "1-based (default) -> converted"
        else:
            raise ValueError("default_base must be '0-based' or '1-based'")

    # Range check (NO clipping)
    if (idx0 < 0).any() or (idx0 >= n_cells).any():
        bad = np.where((idx0 < 0) | (idx0 >= n_cells))[0][:10]
        raise ValueError(
            f"Out-of-range indices after base handling in {path}. "
            f"mode={mode}, n_cells={n_cells}, min={idx0.min()}, max={idx0.max()}, "
            f"examples at rows {bad.tolist()} values {idx0[bad].tolist()}."
        )

    # Optional dedup while preserving first occurrence order
    if dedup:
        _, first_pos = np.unique(idx0, return_index=True)
        first_pos.sort()
        idx0 = idx0[first_pos]

    # Helpful print (you can remove)
    # print(f"[load_indices_csv_robust] {path.name}: {len(idx0)} indices, {mode}, range=[{idx0.min()},{idx0.max()}]")

    return idx0.astype(np.int64)



In [ ]:
# ============================================================
# Preprocessing application (critical)
# ============================================================

def apply_preproc(pp: dict, X):
    """
    Apply the saved preprocessing dict (from training) to a new matrix X.

    pp is either:
      {"type": "svd+scaler", "svd": TruncatedSVD, "scaler": StandardScaler}
    or:
      {"type": "scaler", "scaler": StandardScaler}

    Returns dense (n x k) or (n x p) numpy array depending on pp.
    """
    typ = pp.get("type", None)
    if typ == "svd+scaler":
        svd = pp["svd"]
        scaler = pp["scaler"]
        if not sp.issparse(X):
            # if X became dense, still ok
            Z = svd.transform(X)
        else:
            Z = svd.transform(X)
        Z = scaler.transform(Z)
        return Z

    if typ == "scaler":
        scaler = pp["scaler"]
        X2 = X.toarray() if sp.issparse(X) else np.asarray(X)
        return scaler.transform(X2)

    raise ValueError(f"Unknown preproc type: {typ}")



In [ ]:
# ============================================================
# Metrics
# ============================================================

def evaluate_regression(y_true, y_pred) -> dict:
    from scipy.stats import spearmanr
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    eps = 1e-12

    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    evs  = float(explained_variance_score(y_true, y_pred))

    yt = (y_true - y_true.mean()) / (y_true.std(ddof=0) + eps)
    yp = (y_pred - y_pred.mean()) / (y_pred.std(ddof=0) + eps)
    pearson_r = float(np.clip((yt * yp).mean(), -1.0, 1.0))

    spearman_r = float(spearmanr(y_true, y_pred).statistic)

    return {"rmse": rmse, "mae": mae, "r2": r2, "explained_var": evs,
            "pearson_r": pearson_r, "spearman_r": spearman_r}


In [ ]:
# ============================================================
# Test-time runner
# ============================================================

def predict_on_test_split(
    target_name: str,
    dataset_name: str = "tea",
    split_tag: str = "tea_split3_all_celltypes",
    base_dir: str = "../data",
    splits_dir: str = "../splits",
    models_dir: str = "../models",
    results_dir: str = "../results",
):
    # ---- Paths ----
    model_root = Path(models_dir) / dataset_name / f"{split_tag}_{target_name}_coopreg"
    model_path = model_root / f"{target_name}_coopreg.pkl"
    if not model_path.exists():
        raise FileNotFoundError(f"Missing model: {model_path}")

    out_root = Path(results_dir) / dataset_name / f"{split_tag}_coopreg_{target_name}"
    out_root.mkdir(parents=True, exist_ok=True)

    # ---- Load data (full matrices + y aligned by cell IDs) ----
    bundle = load_regression_data(target_name, base_dir=base_dir)
    n_cells = bundle["rna"].n_obs
    cell_ids = bundle["cell_ids"]
    y_all = bundle["y"]

    # ---- Load TEST indices (robust) ----
    test_idx_path = Path(splits_dir) / f"{split_tag}_test_idx.csv"
    idx_te0 = load_indices_csv_robust(test_idx_path, n_cells=n_cells)
    print(f"[Splits] test idx: n={len(idx_te0)} (from {test_idx_path.name})")

    # ---- Get raw views ----
    X_rna  = get_norm(bundle["rna"])
    X_adtm = get_norm(bundle["adt_minus"])
    X_atac = get_norm(bundle["atac"])

    # Subset to test
    Xte_raw = [X_rna[idx_te0], X_adtm[idx_te0], X_atac[idx_te0]]
    y_true = y_all[idx_te0]
    test_cells = cell_ids[idx_te0].to_numpy()

    # ---- Load trained payload ----
    payload = joblib.load(model_path)
    Thetas = payload["Thetas"]
    preprocs = payload["preprocs"]
    y_mean = float(payload["y_mean"])
    views_order = payload.get("views_order", ["rna", "adt_minus", "atac"])

    if len(Thetas) != 3 or len(preprocs) != 3:
        raise ValueError(f"Expected 3 views. Got Thetas={len(Thetas)} preprocs={len(preprocs)}")

    print("[Model] loaded:", model_path)
    print("[Model] views_order:", views_order)
    print("[Model] theta dims:", [T.shape for T in Thetas])

    # ---- Apply preprocessing (use TRAIN-fitted objects) ----
    Xte = []
    for m in range(3):
        Xm_te = apply_preproc(preprocs[m], Xte_raw[m])
        Xte.append(Xm_te)

    # ---- Predict (add back intercept) ----
    y_pred_c = sum(Xte[m] @ Thetas[m] for m in range(3)).reshape(-1)
    y_pred = y_pred_c + y_mean

    # ---- Evaluate ----
    report = evaluate_regression(y_true, y_pred)
    print("[Test metrics]", report)

    # ---- Save outputs ----
    np.save(out_root / "y_hat.npy", y_pred)
    np.save(out_root / "y_true.npy", y_true)
    np.save(out_root / "idx_test.npy", idx_te0.astype(int))

    pred_df = pd.DataFrame({"cell": test_cells, "y_true": y_true, "y_pred": y_pred})
    pred_df.to_csv(out_root / "test_predictions.csv", index=False)

    report_json = {k: float(v) for k, v in report.items()}
    report_json["run"] = {
        "method": "coopreg",
        "dataset": dataset_name,
        "split_tag": split_tag,
        "target": target_name,
        "views_order": views_order,
        "model_path": str(model_path),
    }
    report_json["params"] = payload.get("params", {})
    report_json["timestamp"] = datetime.now().isoformat(timespec="seconds")

    with open(out_root / "test_metrics.json", "w") as f:
        json.dump(report_json, f, indent=2)

    print("[Saved] outputs under:", out_root)
    print("  - y_hat.npy / y_true.npy / idx_test.npy")
    print("  - test_predictions.csv")
    print("  - test_metrics.json")

    return report, out_root


if __name__ == "__main__":
    predict_on_test_split(target_name="CD45RA")

# %%
